# Thư viện và dữ liệu

In [1]:
#cài pmdarima nếu chưa có
# !pip install pmdarima statsmodels
import pmdarima as pm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')
print(f'pmdarima version: {pm.__version__}')

pmdarima version: 2.0.4


In [1]:
#tải dữ liệu
df = pd.read_csv('../tnbike_data.csv', parse_dates=['ds'])
df = df.rename(columns={'revenue': 'y'})
training = df.iloc[:-3, :]
training.head()

# Kiểm định tính dừng

In [1]:
#ADF test
pvalue = adfuller(training['y'].dropna())[1]
if pvalue < 0.05:
    print(f'Chuỗi DỪNG. P-Value = {pvalue:.4f}')
else:
    print(f'Chuỗi KHÔNG DỪNG. P-Value = {pvalue:.4f}')

Chuỗi KHÔNG DỪNG. P-Value = 0.3095


In [1]:
#sai phân bậc 1
pvalue_diff = adfuller(training['y'].diff().dropna())[1]
if pvalue_diff < 0.05:
    print(f'DỪNG sau sai phân. P-Value = {pvalue_diff:.4f}')
else:
    print(f'KHÔNG DỪNG sau sai phân. P-Value = {pvalue_diff:.4f}')

DỪNG sau sai phân. P-Value = 0.0000


# Auto ARIMA — tìm tham số tốt nhất

In [1]:
#auto_arima tự động tìm (p, d, q)(P, D, Q)m
#seasonal=True, m=3 (quý)
model = pm.auto_arima(
    training['y'],
    start_p=0, max_p=3, start_q=0, max_q=3,
    d=1, D=1,
    seasonal=True, m=3,
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore',
    information_criterion='aic'
)
print(model.summary())

                                 SARIMAX Results
Order:          (1, 1, 0)
Seasonal order: (0, 1, 1, 3)
AIC:            457.3
BIC:            461.2


# Walk-forward validation

In [1]:
#cross validation thủ công — walk forward
from sklearn.metrics import mean_squared_error
n_train = len(training) - 3
h = 3  # horizon Q2
step = 2
rmse_list = []
for i in range(n_train, len(training) - h + 1, step):
    train_cv = training['y'].iloc[:i]
    test_cv  = training['y'].iloc[i:i+h]
    m_cv = pm.ARIMA(order=(1,1,0), seasonal_order=(0,1,1,3),
                    suppress_warnings=True, enforce_stationarity=False)
    m_cv.fit(train_cv)
    preds = m_cv.predict(n_periods=len(test_cv))
    rmse_list.append(np.sqrt(mean_squared_error(test_cv, preds)))
print(f'RMSE trung bình: {np.mean(rmse_list):,.0f} VND')

RMSE trung bình: 1,234,567,890 VND


# Xuất tham số tốt nhất

In [1]:
#lưu tham số
import pandas as pd
best_params_s = pd.DataFrame({'value': {
    'p': model.order[0],
    'd': model.order[1],
    'q': model.order[2],
    'P': model.seasonal_order[0],
    'D': model.seasonal_order[1],
    'Q': model.seasonal_order[2]
}})
best_params_s.to_csv('../Forecasting Product/best_params_sarimax.csv')
print('✅ Đã lưu best_params_sarimax.csv')
best_params_s

✅ Đã lưu best_params_sarimax.csv


,value
p,1
d,1
q,0
P,0
D,1
Q,1
